# Transformer Based Classification

This notebook implements a RoBERTa-based model for emotion classification using the Hugging Face transformers library.

## 0. Setup and Dependencies

In [1]:
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset
import evaluate
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
project_root = Path().absolute().parent
sys.path.append(str(project_root))

from emotion_classifier.data import load_raw_data, prepare_dataset_splits

# Load dataset
train_df, test_df = load_raw_data()
print(f"Loaded dataset with {len(train_df)} samples")

# Create train/validation/test splits if not already done
try:
    train_df, val_df, test_df = prepare_dataset_splits(train_df)
    print(f"Created splits: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")
except Exception as e:
    print(f"Using existing splits: {e}")
    
# Preview the data
display(train_df.head())

# Define label mapping for consistent evaluation
label_mapping = {"Mixed": 0, "Positive": 1, "Negative": 2, "Ambiguous": 3, "Neutral": 4}
id2label = {v: k for k, v in label_mapping.items()}
label2id = label_mapping

print("\nAvailable labels:", list(label_mapping.keys()))

/opt/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded dataset with 3189 samples
Created splits: train=1913, val=638, test=638


,text,primary_emotion,secondary_emotions,meta_emotions,sentiment,interaction_style,intensity,context
1389,"Sometimes, I wonder if I’ll ever truly be happ...",Sadness,"['Regret', 'Loneliness']","['Reflection on fulfillment', 'Commitment to f...",Negative,Supportive,9,Self-Reflection
797,Finally hit my fitness goal today! I’m so prou...,Pride,"['Gratitude', 'Relief']","['Reflection on persistence', 'Commitment to l...",Positive,Empowering,9,Health
824,"Visited the animal shelter today, and seeing a...",Joy,"['Sadness', 'Empathy']","['Reflection on compassion', 'Commitment to he...",Mixed,Supportive,8,Community
1296,I’ve been thinking about how much my perspecti...,Nostalgia,"['Gratitude', 'Sadness']","['Reflection on growth', 'Commitment to living...",Mixed,Reflective,7,Self-Reflection
1630,I’ve been feeling like something is missing in...,Longing,"['Sadness', 'Uncertainty']","['Reflection on fulfillment', 'Commitment to s...",Mixed,Reflective,7,Self-Reflection



Available labels: ['Mixed', 'Positive', 'Negative', 'Ambiguous', 'Neutral']


## 1. Data Preparation
Transform our previously processed dataset into a format suitable for transformer models:
1. Load and split data into train/val/test sets
2. Convert data into HuggingFace Dataset format
3. Initialize RoBERTa model and tokenizer
4. Tokenize text data for all splits
5. Set up label mappings for our emotion classes


In [2]:
def prepare_dataset(df):
    """Convert DataFrame to HuggingFace Dataset format"""
    return Dataset.from_dict({
        'text': df['text'].tolist(),
        'labels': [label_mapping[label] for label in df['sentiment']]
    })

from huggingface_hub import hf_hub_download
filename = hf_hub_download(repo_id="roberta-base", filename="pytorch_model.bin")
print("Downloaded:", filename)

# Convert to HuggingFace datasets
train_dataset = prepare_dataset(train_df)
val_dataset = prepare_dataset(val_df)
test_dataset = prepare_dataset(test_df)

print(f"\nTraining samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

# Load pre-trained model and tokenizer
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_mapping),
    id2label=id2label,
    label2id=label2id,
    force_download=True
)

# Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# Tokenize datasets
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

Downloaded: /Users/sofia.avelino/.cache/huggingface/hub/models--roberta-base/snapshots/e2da8e2f811d1448a5b465c236feacd80ffbac7b/pytorch_model.bin

Training samples: 1913
Validation samples: 638
Test samples: 638


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 638/638 [00:00<00:00, 27489.51 examples/s]


## 2. Model Training and Evaluation
Configure and execute the training process:
1. Define evaluation metrics (F1-score)
2. Calculate class weights to handle imbalanced data
3. Set up training arguments (learning rate, batch size, etc.)
4. Train the model using HuggingFace Trainer
5. Evaluate model performance on test set
6. Save the final model and tokenizer

In [3]:
## 2. Model Training

# Define metrics
metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    return metric.compute(
        predictions=predictions,
        references=labels,
        average="macro"
    )

# Calculate class weights
total_samples = len(train_dataset)
class_counts = train_dataset.select_columns(['labels']).unique('labels')
class_weights = torch.FloatTensor([
    total_samples / (len(train_dataset.filter(lambda x: x['labels'] == i)) * len(class_counts))
    for i in range(len(label_mapping))
])

# Training arguments
training_args = TrainingArguments(
    output_dir="results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="steps",         # Use eval_strategy instead of evaluation_strategy
    save_strategy="steps",         # Use save_strategy instead of save_strategy
    eval_steps=500,               # How often to evaluate
    save_steps=500,               # How often to save
    logging_dir='./logs',         # Directory for storing logs
    logging_steps=100,            # How often to log
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    push_to_hub=False,
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

# Train the model
start_time = time.time()
trainer.train()
end_time = time.time()
training_time = end_time - start_time
print(f"\nTraining time: {training_time:.2f} seconds")
training_duration_base = training_time

Filter: 100%|██████████| 1913/1913 [00:00<00:00, 403667.73 examples/s]


Step,Training Loss,Validation Loss



Training time: 171.33 seconds


In [4]:
## 3. Model Evaluation

# Get predictions
predictions = trainer.predict(tokenized_test)
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

# Print classification report
print("\nClassification Report:")
print(classification_report(
    labels,
    preds,
    target_names=list(label_mapping.keys()),
    zero_division=0
))

# Save the model
output_dir = project_root / "models" / "transformer_model"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"\nModel saved to {output_dir}")


Classification Report:
              precision    recall  f1-score   support

       Mixed       0.71      0.80      0.75       214
    Positive       0.91      0.90      0.90       203
    Negative       0.74      0.82      0.78       127
   Ambiguous       0.31      0.08      0.12        53
     Neutral       0.56      0.56      0.56        41

    accuracy                           0.76       638
   macro avg       0.64      0.63      0.62       638
weighted avg       0.74      0.76      0.74       638


Model saved to /Users/sofia.avelino/Documents/Emotion_Classifier/models/transformer_model


## 3. Parameter-efficient fine-tuning
Apply LoRA (Low-Rank Adaptation) for efficient fine-tuning of the pre-trained model:

1. Configure a LoraConfig object for sequence classification with specified rank, alpha, and dropout.

2. Wrap the base model with get_peft_model to enable parameter-efficient fine-tuning.

3. Display the number of trainable parameters for transparency and reproducibility.

4. Repeat the training and evaluation steps of the previous section.

5. Compare the results with and without parameter-efficient fine-tuning.

In [5]:
from peft import LoraConfig, TaskType, get_peft_model

peft_config = LoraConfig(task_type = TaskType.SEQ_CLS, inference_mode=False, r=8, lora_alpha=32, lora_dropout=0.1)

model_peft = get_peft_model(model, peft_config)
model_peft.print_trainable_parameters()

trainable params: 889,349 || all params: 125,538,826 || trainable%: 0.7084


In [6]:

#Model training
# Define metrics
metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    return metric.compute(
        predictions=predictions,
        references=labels,
        average="macro"
    )

# Calculate class weights
total_samples = len(train_dataset)
class_counts = train_dataset.select_columns(['labels']).unique('labels')
class_weights = torch.FloatTensor([
    total_samples / (len(train_dataset.filter(lambda x: x['labels'] == i)) * len(class_counts))
    for i in range(len(label_mapping))
])

# Training arguments
training_args = TrainingArguments(
    output_dir="results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="steps",         # Use eval_strategy instead of evaluation_strategy
    save_strategy="steps",         # Use save_strategy instead of save_strategy
    eval_steps=500,               # How often to evaluate
    save_steps=500,               # How often to save
    logging_dir='./logs',         # Directory for storing logs
    logging_steps=100,            # How often to log
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    push_to_hub=False,
)

# Initialize trainer
trainer = Trainer(
    model=model_peft,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

# Train the model
start_time = time.time()
trainer.train()
end_time = time.time()
training_time = end_time - start_time
print(f"\nTraining time: {training_time:.2f} seconds")
training_duration_peft = training_time

Filter: 100%|██████████| 1913/1913 [00:00<00:00, 395978.07 examples/s]
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss,Validation Loss



Training time: 120.26 seconds


In [ ]:
## 3. Model Evaluation

# Get predictions
predictions = trainer.predict(tokenized_test)
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

# Print classification report
print("\nClassification Report:")
print(classification_report(
    labels,
    preds,
    target_names=list(label_mapping.keys()),
    zero_division=0
))

class_peft = classification_report(
    labels,
    preds,
    target_names=list(label_mapping.keys()),
    zero_division=0
)

# Save the model
output_dir = project_root / "models" / "transformer_model_peft"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"\nModel saved to {output_dir}")


Classification Report:
              precision    recall  f1-score   support

       Mixed       0.74      0.78      0.76       214
    Positive       0.90      0.89      0.89       203
    Negative       0.73      0.80      0.76       127
   Ambiguous       0.40      0.23      0.29        53
     Neutral       0.55      0.59      0.56        41

    accuracy                           0.76       638
   macro avg       0.66      0.65      0.65       638
weighted avg       0.75      0.76      0.75       638



TypeError: missing a required argument: 'y_true'

In [9]:
import pandas as pd

# Replace with your actual F1-scores and durations
results = {
    "Class": ["Mixed", "Positive", "Negative", "Ambiguous", "Neutral", "Macro F1", "Weighted F1", "Accuracy", "Training Time (s)"],
    "Baseline F1": [0.74, 0.90, 0.77, 0.09, 0.65, 0.63, 0.74, 0.76,  training_duration_base], 
    "PEFT F1":     [0.77, 0.91, 0.76, 0.27, 0.62, 0.67, 0.76, 0.77,  training_duration_peft],    
}

df = pd.DataFrame(results)

# Add Delta column (PEFT - Baseline)
df["Δ (PEFT - Baseline)"] = df["PEFT F1"] - df["Baseline F1"]

# Print formatted table
print(df.to_string(index=False, float_format="%.2f"))


            Class  Baseline F1  PEFT F1  Δ (PEFT - Baseline)
            Mixed         0.74     0.77                 0.03
         Positive         0.90     0.91                 0.01
         Negative         0.77     0.76                -0.01
        Ambiguous         0.09     0.27                 0.18
          Neutral         0.65     0.62                -0.03
         Macro F1         0.63     0.67                 0.04
      Weighted F1         0.74     0.76                 0.02
         Accuracy         0.76     0.77                 0.01
Training Time (s)       171.33   120.26               -51.07
